# Web Scraping Fundamentals

**Web scraping** is the automated extraction of data from websites. Instead of copying data by hand, we write code that fetches pages, extracts the useful parts, and saves them as structured data (CSV, Excel, JSON).

This notebook focuses on **static pages** — pages where the content is already in the HTML. We'll scrape the public demo site [scrapethissite.com](https://www.scrapethissite.com) and a card-shop page, and save our results.

**What you will learn:**

- Fetch a web page with the `requests` library
- Parse HTML with `BeautifulSoup` (`find`, `find_all`, `.text`, `.get`)
- Turn scraped data into a clean pandas DataFrame
- Scrape paginated tables (multi-page listings)
- Extract image URLs and download them
- Know when you need Selenium (dynamic pages)

> **Be a good scraper:** only scrape sites that allow it, respect `robots.txt` and the site's terms, keep request rates low, and identify your bot. This notebook uses a public practice site designed for scraping.

## The Web-Scraping Toolbox

| Library | Job |
|---------|-----|
| `requests` | Download web pages (HTTP) |
| `BeautifulSoup` (`bs4`) | Parse / navigate the HTML |
| `pandas` | Store the results in tables |
| `selenium` | Automate a real browser (dynamic pages) |
| `tqdm` | Progress bars for long loops |

## Setup: Installing the Libraries

Run once in your terminal (if not already installed):

```
pip install requests beautifulsoup4 pandas selenium tqdm lxml
```

> `lxml` is used by `pd.read_html()` to parse HTML tables quickly (it's an optional pandas dependency).

Then import everything:

In [ ]:
import requests as r          # fetch pages
from bs4 import BeautifulSoup as bs   # parse HTML
import pandas as pd          # structure the results
import os                    # file/folder helpers

## The Scraping Workflow

Every scraping project follows the same 4 steps:

1. **Fetch** — download the page (or use a saved offline copy)
2. **Parse** — locate the interesting elements in the HTML
3. **Structure** — collect the data into a pandas DataFrame
4. **Save** — export to Excel / CSV / JSON

> This folder already contains **offline copies** of the pages we need (`htmlFiles/`, `ScrapeThisSiteOffline.html`, `deskShopOffline.html`). You can parse those without internet; the fetch cells are optional (and wrapped so they degrade gracefully offline).

## Step 1: Fetch a Page

`requests.get(url)` downloads the page. `response.status_code` tells us if it worked (`200` = OK). We save the HTML to disk so we can parse it later — even offline.

> **Tip:** always check `response.raise_for_status()` / `status_code` before parsing — servers return error pages that are useless to parse.

In [ ]:
try:
    # The practice site's "simple" page lists every country in a table
    response = r.get("https://www.scrapethissite.com/pages/simple/", timeout=10)
    response.raise_for_status()

    # Save a copy of the page for offline parsing
    with open("htmlFiles/ScrapeThisSiteResponse.html", "w", encoding="utf-8") as file:
        file.write(str(bs(response.content, "html.parser")))
    print("Fetched and saved OK -> htmlFiles/ScrapeThisSiteResponse.html")
except Exception as e:
    print("No internet? Skipping the fetch — an offline copy already exists.")
    print(e)

## Step 2: Parse HTML with BeautifulSoup

BeautifulSoup turns raw HTML into a **navigable tree**. Key methods:

| Method | What it does |
|--------|--------------|
| `soup.find("tag")` | First matching tag |
| `soup.find_all("tag")` | All matching tags |
| `soup.find_all("h3", class_="country-name")` | Filter by class |
| `soup.select("h3.country-name")` | CSS-selector style |
| `tag.text.strip()` | Visible text of an element |
| `tag.get("src")` | Value of an attribute |

Each country on this page is an `<h3 class="country-name">`. We parse the saved file — **no network needed**:

In [ ]:
# Parse the saved page (offline)
with open("htmlFiles/ScrapeThisSiteResponse.html", "r", encoding="utf-8") as file:
    soup = bs(file.read(), "html.parser")

# Find every country name heading
countryNames = soup.find_all("h3", class_="country-name")
print("Found", len(countryNames), "countries")

for name in countryNames[:5]:
    print(" -", name.text.strip())

## Step 3: Structure the Data with pandas

A page is full of elements, not data. We collect the matching elements for each field **in the same order** (element *i* of every list = country *i*), then build a dictionary of lists — which pandas turns straight into a DataFrame:

In [ ]:
# Grab the other three fields for every country (same order!)
countryCapitals    = soup.find_all("span", class_="country-capital")
countryPopulations = soup.find_all("span", class_="country-population")
countryAreas       = soup.find_all("span", class_="country-area")

# Dictionary of lists -> pandas builds the DataFrame from it
CountryInfo = {
    "Country Name":      [],
    "Country Capital":   [],
    "Country Population":[],
    "Country Area":      [],
}

for i in range(len(countryNames)):
    CountryInfo["Country Name"].append(countryNames[i].text.strip())
    CountryInfo["Country Capital"].append(countryCapitals[i].text.strip())
    CountryInfo["Country Population"].append(countryPopulations[i].text.strip())
    CountryInfo["Country Area"].append(countryAreas[i].text.strip())

# Create the DataFrame and export it
countryInfoDataFrame = pd.DataFrame(CountryInfo)
countryInfoDataFrame.to_excel("CountryData.xlsx", index=False)
print("Saved CountryData.xlsx —", countryInfoDataFrame.shape[0], "countries")

In [ ]:
countryInfoDataFrame

## Step 4: Scraping Paginated Tables

Large listings are split across **pages** (`?page_num=1`, `?page_num=2`, ...). The trick is to loop over page numbers and save each page. The `forms` section of the practice site has 24 pages of hockey-team data — offline copies of all 24 are already in `htmlFiles/`:

In [ ]:
# Live demo: fetch the first 3 pages (a full run would do all 24).
# Offline students can skip this — htmlFiles/ already has all 24 saved.
try:
    for i in range(1, 4):
        response = r.get(f"https://www.scrapethissite.com/pages/forms/?page_num={i}", timeout=10)
        soup = bs(response.content, "html.parser")
        with open(f"htmlFiles/scrapethissitePageNo{i}.html", "w", encoding="utf-8") as file:
            file.write(str(soup.prettify()))
        print(f"Saved page {i}")
except Exception as e:
    print("No internet? Using the offline copies already in htmlFiles/.")
    print(e)

### Parsing a Table Directly with `pd.read_html()`

Many sites put tabular data in an actual HTML `<table>`. pandas can read those **directly** — no manual element-finding needed:

In [ ]:
# read_html() finds every <table> in the file and returns one DataFrame per table
tables = pd.read_html("htmlFiles/scrapethissitePageNo1.html")
print("Tables found on the page:", len(tables))

hockey_df = tables[0]
hockey_df.head()

## Step 5: Scraping Images

Downloading images follows the same pattern:

1. Find the `<img>` tags (here: card artworks under `/img/card/...`)
2. Collect each image's **name** (`alt`) and **URL** (`src`)
3. Loop over the URLs, download each one, and save to disk

We parse the offline copy of the card-shop page first (no network):

In [ ]:
# Parse the offline copy of the card-shop page
with open("deskShopOffline.html", "r", encoding="utf-8") as file:
    soup = bs(file.read(), "html.parser")

# Card artwork URLs all start with /img/card — collect a small sample
cardImages = [img for img in soup.find_all("img")
              if (img.get("src") or "").startswith("/img/card")]
print("Card images on the page:", len(cardImages))

ImageUrlData = {"Name": [], "Url": []}
for img in cardImages[:8]:  # demo: first 8 cards
    name = str(img.get("alt")).strip()
    ImageUrlData["Name"].append(name if name else f"card_{len(ImageUrlData['Name'])}")
    ImageUrlData["Url"].append(str(img.get("src")).strip())

image_df = pd.DataFrame(ImageUrlData)
image_df

### Downloading the Images

Now loop over the URLs and save each file to `ScrappedImages/`. `tqdm` gives us a progress bar. **This cell needs internet** — if you're offline, the already-downloaded images in `ScrappedImages/` are there to inspect.

In [ ]:
from tqdm import tqdm

os.makedirs("ScrappedImages", exist_ok=True)
baseurl = "https://www.deckshop.pro"   # the site the images come from

try:
    for i in tqdm(range(len(ImageUrlData["Url"])), desc="Downloading images"):
        img_url = ImageUrlData["Url"][i]
        response = r.get(f"{baseurl}{img_url}", timeout=15)
        if response.status_code == 200:
            safe_name = ImageUrlData["Name"][i].replace("/", "_").replace(" ", "_")
            with open(f"ScrappedImages/{safe_name}.png", "wb") as f:
                f.write(response.content)
    print("Done — check the ScrappedImages/ folder!")
except Exception as e:
    print("No internet? Skipping the download (existing images stay as-is).")
    print(e)

## When Do We Need Selenium?

`requests` + BeautifulSoup only see the raw HTML. Some sites render content with **JavaScript** — the data isn't in the HTML at all. For those you need a real browser (Selenium) that executes the JavaScript first.

That's a whole topic of its own — it's covered in detail in **`06_Web_Automation/01_Selenium_Automation.ipynb`**. For now, just confirm Selenium is available on this machine:

In [ ]:
try:
    from selenium import webdriver
    print("Selenium is installed — ready for the automation notebook.")
except ImportError:
    print("Selenium not installed. Run: pip install selenium")

## 🎯 Key Takeaways

- Scraping = **fetch → parse → structure → save**.
- `requests.get()` downloads; always check `status_code`.
- Save pages to disk once, then parse offline — faster and gentler on servers.
- `BeautifulSoup` finds elements with `find` / `find_all` and CSS classes.
- Aligned lists of extracted values become a pandas DataFrame via a dict.
- Paginated sites: loop over `page_num`, save each page.
- `pd.read_html()` parses HTML `<table>`s directly into DataFrames.
- Images: collect `src` URLs, then download in a loop with a progress bar.
- Scrape **ethically**: respect terms, `robots.txt`, and rate limits.

## 🏋️ Practice Exercises

1. Change Step 2 to print the **10 longest** country names (`.text` + `sorted(..., key=len)`).
2. On the countries page, `find_all("div", class_="col-md-4")` — how many country cards are there per row?
3. Parse `htmlFiles/scrapethissitePageNo1.html` with `pd.read_html()` and find the team with the most **wins**.
4. Scrape all **24** pages of the forms section (set the loop to `range(1, 25)`) and build one combined DataFrame.
5. In the image section, extend the sample to **all** card images (remove the `[:8]`) and count how many download successfully.

## 🚀 Next Steps

- **`02_Scraping_Advanced.ipynb`** — deeper parsing, real-world sites, and cleanups.
- **`06_Web_Automation/01_Selenium_Automation.ipynb`** — JavaScript pages, logins, and browser automation.
- **`03_Pandas_Data_Analysis`** — analyze everything you scrape with pandas.